# Keyword context / KWIC export (v2)

For every keyword hit (environmental lexicon + technology lexicon), pulls a ~5-sentence
window of surrounding context (2 before + keyword sentence + 2 after), tagged with
title/author/year/era. Overlapping/adjacent windows merge into one passage with a
`keywords_found` column, rather than many near-duplicate rows.

**Notebook, not a script, on purpose** -- same reason as `SF_novel_keyness_v2.ipynb`:
`scripts/keyword_context_v2.py` ran to completion and printed correct numbers, but the
output never actually landed on disk, and the only way to find that out was after the
whole multi-hour corpus pass had already finished. This version separates the three
costs cleanly:

1. **OCR cleaning** (`docs = load_docs_keep_id(...)`) -- the genuinely slow step, run once.
2. **Passage extraction** -- sentence-splitting + keyword matching + window-merging
   against the *already-cleaned* text in memory. Much cheaper than step 1 (no more
   per-page dictionary-rate scoring or disk reads), so re-running it after a failure
   is a small cost, not a multi-hour one.
3. **Write + verify** -- writes the CSVs, then re-reads them fresh from disk and checks
   the row counts match what was computed in memory. This is the check that was missing
   before: a `print(f"{n} rows")` right after `.to_csv()` only proves the in-memory data
   was correct, never that the write actually persisted.

No cap on occurrences -- every keyword hit is captured. `tables/book_summary.csv` reports,
per book per lexicon, what percent of the book's own words ended up in exported passages,
and flags (prints + a `flagged` column) anything at or above `FLAG_THRESHOLD_PCT`.

**2026-09-24**: passages are shuffled (globally, not just within-book, fixed seed for
reproducibility) before writing, and `FLAG_THRESHOLD_PCT` raised from 30 to 40. Both
changes go together: non-consumptive use is about not exporting anything that could
substitute for reading the book, and a shuffled, out-of-sequence set of passages is a
much weaker case for that than the same passages presented in original reading order --
so the higher threshold is reasonable specifically because the ordering problem (which
was the sharper risk) is fixed at the same time, not as an independent relaxation.

Must run **inside the capsule, in secure mode**.

## Imports

In [ ]:
import csv
import json
import os
import random
import re
from collections import Counter
from multiprocessing import get_context
from pathlib import Path

import pandas as pd
from nltk.tokenize import TreebankWordTokenizer, sent_tokenize

## Check the sentence tokenizer is available

Secure mode has no network -- if this fails, run
`python3 -c "import nltk; nltk.download('punkt'); nltk.download('punkt_tab')"` from
maintenance mode (network on) first, then switch to secure mode and retry.

In [ ]:
try:
    sent_tokenize("Checking that the sentence tokenizer's data is available. This is only a test.")
    print("tokenizer OK")
except LookupError as e:
    raise SystemExit(
        "NLTK sentence-tokenizer data not found, and secure mode has no network to fetch it. "
        "Run this from maintenance mode BEFORE switching to secure mode:\n"
        "  python3 -c \"import nltk; nltk.download('punkt'); nltk.download('punkt_tab')\"\n"
        f"Original error: {e}"
    )

## OCR cleaning

Copied verbatim from `SF_word2vec_eras_v2.ipynb` (cell `46a9ca82`) / `scripts/keyword_context_v2.py`.

In [ ]:
_norm_ws = re.compile(r"\s+")
_only_number = re.compile(r"^\s*[\divxlcIVXLC]+\s*$")
_hyphen_break = re.compile(r"(\w)-\s*\n\s*(\w)")
_word = re.compile(r"[A-Za-z']+")


def discover_volumes(base_dir, ids=None):
    vols = {}
    for d in sorted(p for p in Path(base_dir).iterdir() if p.is_dir() and not p.name.startswith(".")):
        if ids is not None and d.name not in ids:
            continue
        pages = sorted(d.glob("*.txt"))
        if pages:
            vols[d.name] = pages
    return vols


def _norm_line(line):
    return _norm_ws.sub(" ", re.sub(r"\d+", "", line)).strip().lower()


def _volume_vocab(page_paths):
    words = set()
    for p in page_paths:
        text, _ = _hyphen_break.subn(r"\1\2", p.read_text(encoding="utf-8", errors="replace"))
        words.update(w.lower() for w in _word.findall(text) if len(w) > 1)
    return words


def build_dictionary(volumes, min_vols, procs=None):
    df = Counter()
    with get_context("fork").Pool(procs or min(16, os.cpu_count() or 1)) as pool:
        for vocab in pool.imap_unordered(_volume_vocab, list(volumes.values()), chunksize=8):
            df.update(vocab)
    return {w for w, c in df.items() if c >= min_vols}


def page_dict_rate(text, dictionary):
    words = [w.lower() for w in _word.findall(text) if len(w) > 1]
    return sum(1 for w in words if w in dictionary) / len(words) if words else 0.0


def clean_volume(page_paths, dictionary, running_head_min_pages, running_head_frac,
                  running_head_max_chars, page_min_dict_rate):
    pages = [p.read_text(encoding="utf-8", errors="replace") for p in page_paths]
    line_pages = Counter()
    per_page_lines = []
    for text in pages:
        lines = text.split("\n")
        per_page_lines.append(lines)
        line_pages.update({_norm_line(l) for l in lines if 0 < len(l.strip()) <= running_head_max_chars})
    thresh = max(running_head_min_pages, int(running_head_frac * len(pages)))
    heads = {l for l, c in line_pages.items() if c >= thresh and l}
    kept = []
    for lines in per_page_lines:
        out = []
        for l in lines:
            if (len(l.strip()) <= running_head_max_chars and _norm_line(l) in heads) or _only_number.match(l):
                continue
            out.append(l)
        page_text, _ = _hyphen_break.subn(r"\1\2", "\n".join(out))
        if dictionary is not None and page_min_dict_rate and page_dict_rate(page_text, dictionary) < page_min_dict_rate:
            continue
        kept.append(page_text)
    text, _ = _hyphen_break.subn(r"\1\2", "\n".join(kept))
    return text


def load_docs_keep_id(base_dir, dictionary, ids=None, **clean_kwargs):
    docs = {}
    for htid, pages in discover_volumes(base_dir, ids).items():
        docs[htid] = clean_volume(pages, dictionary, **clean_kwargs)
    return docs

## Tokenization for keyword matching

Same `TOKEN_RE` as `SF_word2vec_eras_v2.ipynb` (cell `be5f5689`).

In [ ]:
_tokenizer = TreebankWordTokenizer()
TOKEN_RE = re.compile(r"^[a-z]+(?:'[a-z]+)?$")


def clean_tokens(tokens):
    return [t for t in tokens if TOKEN_RE.match(t) and len(t) > 1]


def sentences_with_tokens(text):
    """Raw (readable, original-case) sentences paired with their lowercased match-tokens."""
    raw_sents = sent_tokenize(text)
    out = []
    for s in raw_sents:
        toks = set(clean_tokens(_tokenizer.tokenize(s.lower())))
        out.append((s, toks))
    return out

## Era assignment

In [ ]:
def assign_era(year, cutoffs=(1962, 1972), labels=("era_a", "era_b", "era_c")):
    for cutoff, label in zip(cutoffs, labels):
        if year < cutoff:
            return label
    return labels[-1]

## Metadata loading

Same Ace-Doubles handling as `SF_novel_keyness_v2.ipynb` -- `year` identical within every
duplicate-`htid` group (checked), `title`/`author` joined with `" / "`.

In [ ]:
def load_metadata(path):
    df = pd.read_csv(path)
    df["htid"] = df["htid"].astype(str)

    def join_unique(values):
        return " / ".join(dict.fromkeys(str(v) for v in values))

    grouped = df.groupby("htid").agg(
        title=("title", join_unique),
        author=("author", join_unique),
        year=("year", "first"),
    )
    return grouped.to_dict("index")

## Lexicons (reviewed 2026-09-23)

Environmental: 101 words (same as `SF_novel_keyness_v2.ipynb` -- `war` added,
`power`/`cycle`/`ice`/`cistern`/`culvert` dropped). Technology: 24 words, WordNet-audited,
zero overlap with the environmental list.

In [ ]:
ENV_WORD_GROUPS = {
    "landscape_baseline": ["river", "creek", "stream", "water", "forest", "nature", "wilderness", "jungle",
                            "ocean", "landscape", "levee", "dam", "reservoir", "estuary", "wetland",
                            "marsh", "watershed"],
    "ecology_concept": ["ecology", "ecosystem", "environment", "biosphere", "habitat", "balance"],
    "contamination": ["contamination", "waste", "smog", "fumes", "chemical", "pesticide", "insecticide",
                       "pollutant", "exhaust", "toxic", "polluted", "pollution"],
    "waste_infrastructure": ["sewer", "sewage", "drainage", "effluent", "runoff", "wastewater",
                              "cesspool", "sludge", "septic", "plumbing"],
    "population_scarcity": ["overpopulation", "population", "famine", "scarcity", "starvation", "resource", "drought"],
    "energy": ["oil", "fuel", "energy", "coal"],
    "nuclear_atomic": ["radiation", "radioactive", "fallout", "nuclear", "atomic", "bomb", "meltdown"],
    "cosmic_natural_causation": ["solar", "cosmic", "celestial", "geological", "planetary"],
    "human_agency": ["mankind", "humanity", "civilization", "industrial", "war"],
    "disaster_collapse": ["wasteland", "extinction", "collapse", "barren", "dying", "decay", "catastrophe",
                           "apocalypse", "plague"],
    "climate_weather": ["climate", "weather", "warming", "greenhouse", "atmosphere", "temperature",
                         "flood", "flooding", "storm", "hurricane", "glacier", "carbon", "ozone"],
    "space_earth_framing": ["earth", "homeworld", "colony", "frontier", "terraform", "alien"],
}

TECH_WORD_GROUPS = {
    "automation_machinery": ["machinery", "mechanical", "automaton", "automation", "automated"],
    "artificial_beings": ["robot", "android", "cyborg"],
    "computing_electronics": ["computer", "cybernetic", "electronic", "circuitry"],
    "engineering_industry": ["engineering", "engineer", "technology", "technological", "factory"],
    "synthetic_material": ["synthetic", "artificial"],
    "space_energy_tech": ["rocket", "spacecraft", "satellite", "laser", "reactor"],
}


def word_to_group_map(word_groups):
    return {w: g for g, ws in word_groups.items() for w in ws}


env_words = set(word_to_group_map(ENV_WORD_GROUPS))
tech_words = set(word_to_group_map(TECH_WORD_GROUPS))
env_word_to_group = word_to_group_map(ENV_WORD_GROUPS)
tech_word_to_group = word_to_group_map(TECH_WORD_GROUPS)
print(f"env: {len(env_words)} words, tech: {len(tech_words)} words, overlap: {env_words & tech_words}")

## Window-merging KWIC extraction

In [ ]:
def extract_passages(sent_list, lexicon_words, word_to_group, sentences_before=2, sentences_after=2):
    """sent_list: [(raw_sentence, token_set), ...] for one novel.
    Returns list of dicts: one per merged passage."""
    n = len(sent_list)
    hits = []
    for i, (_, toks) in enumerate(sent_list):
        matched = toks & lexicon_words
        if matched:
            hits.append((i, matched))
    if not hits:
        return []

    windows = []
    for i, matched in hits:
        start = max(0, i - sentences_before)
        end = min(n - 1, i + sentences_after)
        windows.append([start, end, set(matched)])

    merged = [windows[0]]
    for start, end, matched in windows[1:]:
        last = merged[-1]
        if start <= last[1] + 1:
            last[1] = max(last[1], end)
            last[2] |= matched
        else:
            merged.append([start, end, set(matched)])

    passages = []
    for start, end, matched in merged:
        context = " ".join(sent_list[j][0] for j in range(start, end + 1))
        groups = sorted({word_to_group[w] for w in matched})
        passages.append({
            "keywords_found": ", ".join(sorted(matched)),
            "groups_found": ", ".join(groups),
            "context": context,
            "n_sentences": end - start + 1,
            "sent_start": start,
            "sent_end": end,
            "context_words": len(context.split()),
        })
    return passages

## Config

In [ ]:
TEXT_DIR = "/media/secure_volume/fa50b375-3216-4edd-a685-98488562b723"
METADATA_CSV = "/home/dcuser/Desktop/Clifi-htrc/notebooks/metadata_august2026.csv"
OUT_DIR = Path("/media/secure_volume/out_keyword_context_v2")
(OUT_DIR / "tables").mkdir(parents=True, exist_ok=True)

SENTENCES_BEFORE = 2
SENTENCES_AFTER = 2
FLAG_THRESHOLD_PCT = 40.0  # raised from 30 2026-09-24 -- reasonable given the shuffle below:
# a scrambled, out-of-sequence set of passages is a much weaker "could replace reading
# the book" case than the same percentage presented in original reading order would be.
SHUFFLE_SEED = 20260924    # fixed seed so the shuffle is reproducible if regenerated
MAX_CSV_BYTES = 900_000_000   # split into a new part file before hitting this, under the 1GB export limit
DICT_MIN_VOLS = 10
PAGE_MIN_DICT_RATE = 0.55
RUNNING_HEAD_MIN_PAGES = 3
RUNNING_HEAD_FRAC = 0.05
RUNNING_HEAD_MAX_CHARS = 60

## Load metadata

In [ ]:
meta_by_id = load_metadata(METADATA_CSV)
print(f"{len(meta_by_id):,} unique htids after grouping Ace Doubles / omnibus scans")

## Discover volumes + build corpus dictionary

In [ ]:
all_volumes = discover_volumes(TEXT_DIR)
print(f"found {len(all_volumes):,} volumes")

dictionary = build_dictionary(all_volumes, DICT_MIN_VOLS)
print(f"dictionary: {len(dictionary):,} words appear in >= {DICT_MIN_VOLS} volumes")

## Clean every volume -- THE SLOW STEP

Once this finishes, `docs` stays in kernel memory. Everything below is much cheaper and
safe to re-run without paying this cost again.

In [ ]:
docs = load_docs_keep_id(
    TEXT_DIR, dictionary,
    running_head_min_pages=RUNNING_HEAD_MIN_PAGES,
    running_head_frac=RUNNING_HEAD_FRAC,
    running_head_max_chars=RUNNING_HEAD_MAX_CHARS,
    page_min_dict_rate=PAGE_MIN_DICT_RATE,
)
print(f"loaded {len(docs):,} cleaned documents")

## Extract passages from the already-cleaned text

Sentence-splits + matches keywords + merges windows for every novel. Much cheaper than
the cleaning step above (no more disk reads or per-page dictionary scoring), so if
verification fails later, re-running from here is a small cost, not a multi-hour one.
Builds everything in memory first -- `env_rows`, `tech_rows`, `book_summary_rows` --
nothing is written to disk yet.

In [ ]:
env_rows = []
tech_rows = []
book_summary_rows = []
flagged_books = []

for count, htid in enumerate(docs, 1):
    text = docs[htid]
    book_total_words = len(text.split())
    if book_total_words == 0:
        continue

    sent_list = sentences_with_tokens(text)
    info = meta_by_id.get(htid, {})
    year = info.get("year")
    row_meta = {
        "htid": htid,
        "title": info.get("title"),
        "author": info.get("author"),
        "year": year,
        "era": assign_era(year) if pd.notna(year) else None,
    }
    book_row = {**row_meta, "book_total_words": book_total_words}

    for label, words, word_to_group, out_rows in [
        ("env", env_words, env_word_to_group, env_rows),
        ("tech", tech_words, tech_word_to_group, tech_rows),
    ]:
        passages = extract_passages(sent_list, words, word_to_group, SENTENCES_BEFORE, SENTENCES_AFTER)
        exported_words = sum(p["context_words"] for p in passages)
        pct = round(100 * exported_words / book_total_words, 2)
        flagged = pct >= FLAG_THRESHOLD_PCT
        book_row[f"{label}_passages"] = len(passages)
        book_row[f"{label}_exported_words"] = exported_words
        book_row[f"{label}_pct_exported"] = pct
        book_row[f"{label}_flagged"] = flagged
        if flagged:
            flagged_books.append((label, row_meta["title"], row_meta["author"], row_meta["year"], pct))
        for p in passages:
            out_rows.append({**p, **row_meta})

    book_summary_rows.append(book_row)
    if count % 500 == 0:
        print(f"  ... {count:,}/{len(docs):,} novels processed")

print(f"done: {len(env_rows):,} env passages, {len(tech_rows):,} tech passages, "
      f"{len(book_summary_rows):,} books summarized")

## Print flagged books

Report-and-flag, not a silent cap -- review these before releasing anything.

In [ ]:
if flagged_books:
    print(f"*** {len(flagged_books)} lexicon/book combinations flagged at >= {FLAG_THRESHOLD_PCT}% "
          f"of the book's own words exported -- REVIEW BEFORE RELEASING: ***")
    for label, title, author, year, pct in sorted(flagged_books, key=lambda x: -x[4]):
        print(f"  [{label}] {pct:.1f}%  {title} ({author}, {year})")
else:
    print(f"no book exceeded the {FLAG_THRESHOLD_PCT}% flag threshold for either lexicon.")

## Which words are driving the flagged books?

`book_summary.csv` only has the *aggregate* percentage exported per book per lexicon --
not which specific word is responsible. This counts, for each word, how many exported
passages it appears in *among the flagged books only* -- a small, safe-to-export
aggregate table (word + a count, no text) that answers "which seed words should we
consider narrowing" without exporting anything from the actual passage CSVs to look at.

Note on what the count means: since a merged passage's `keywords_found` can list several
words at once, this counts *how many flagged-book passages contain word W*, not W's raw
occurrence count -- a passage where "water" appears three times in a row still only
counts once here. That's still a fair proxy for which words are driving flagged books'
volume, just not a literal frequency count.

In [ ]:
flagged_htids = {
    "env": {r["htid"] for r in book_summary_rows if r["env_flagged"]},
    "tech": {r["htid"] for r in book_summary_rows if r["tech_flagged"]},
}


def keyword_counts_in_flagged(rows, flagged_ids):
    counter = Counter()
    for row in rows:
        if row["htid"] not in flagged_ids:
            continue
        for w in row["keywords_found"].split(", "):
            counter[w] += 1
    return counter


flagged_word_rows = []
for lexicon, rows, flagged_ids in [("env", env_rows, flagged_htids["env"]), ("tech", tech_rows, flagged_htids["tech"])]:
    counts = keyword_counts_in_flagged(rows, flagged_ids)
    for word, n in counts.items():
        flagged_word_rows.append({"lexicon": lexicon, "word": word, "passages_in_flagged_books": n})

flagged_word_counts_df = pd.DataFrame(flagged_word_rows).sort_values(
    ["lexicon", "passages_in_flagged_books"], ascending=[True, False]
)
print(f"{len(flagged_htids['env'])} flagged env books, {len(flagged_htids['tech'])} flagged tech books")
print(flagged_word_counts_df.to_string(index=False))

In [ ]:
flagged_word_counts_path = OUT_DIR / "tables" / "flagged_word_counts.csv"
flagged_word_counts_df.to_csv(flagged_word_counts_path, index=False)
print(f"wrote {flagged_word_counts_path}")

reread_fwc = pd.read_csv(flagged_word_counts_path)
assert len(reread_fwc) == len(flagged_word_counts_df), (
    f"MISMATCH: wrote {len(flagged_word_counts_df)} rows but disk shows {len(reread_fwc)}"
)
print(f"VERIFIED: {flagged_word_counts_path} has {len(reread_fwc):,} rows on disk, matches expected {len(flagged_word_counts_df):,}.")

## Write the CSVs (auto-splitting) + the book summary

Safe to re-run this cell (and the verify cell after it) as many times as needed --
`env_rows`, `tech_rows`, and `book_summary_rows` are already sitting in memory.

In [ ]:
CONTEXT_FIELDNAMES = ["keywords_found", "groups_found", "context", "n_sentences", "context_words",
                       "sent_start", "sent_end", "htid", "title", "author", "year", "era"]

# Shuffle globally (not just within-book) before writing: currently env_rows/tech_rows are
# in sentence order within each book, and grouped by book (sorted by htid) across the whole
# list -- meaning the unshuffled file would let someone read a coherent, in-order excerpt
# trail through any one book. Shuffling breaks that while keeping every row's own
# title/author/year/keywords_found intact, so nothing about filtering, grouping, or
# reading any single passage in isolation changes.
random.Random(SHUFFLE_SEED).shuffle(env_rows)
random.Random(SHUFFLE_SEED).shuffle(tech_rows)


def write_rolling_csv(rows, out_dir, base_name, fieldnames, max_bytes):
    """Writes rows across part files (part1, part2, ...), splitting before max_bytes."""
    part = 1
    parts_written = []

    def open_part(n):
        path = out_dir / f"{base_name}_part{n}.csv"
        f = open(path, "w", newline="", encoding="utf-8")
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        parts_written.append(path)
        return f, writer

    f, writer = open_part(part)
    for row in rows:
        writer.writerow(row)
        if f.tell() >= max_bytes:
            f.close()
            part += 1
            f, writer = open_part(part)
    f.close()
    return parts_written


env_parts = write_rolling_csv(env_rows, OUT_DIR, "environment_context", CONTEXT_FIELDNAMES, MAX_CSV_BYTES)
print(f"environment_context: {len(env_rows):,} passages across {len(env_parts)} file(s)")
for p in env_parts:
    print(f"  {p} ({p.stat().st_size / 1e6:.1f} MB)")

tech_parts = write_rolling_csv(tech_rows, OUT_DIR, "technology_context", CONTEXT_FIELDNAMES, MAX_CSV_BYTES)
print(f"technology_context: {len(tech_rows):,} passages across {len(tech_parts)} file(s)")
for p in tech_parts:
    print(f"  {p} ({p.stat().st_size / 1e6:.1f} MB)")

summary_df = pd.DataFrame(book_summary_rows)
summary_path = OUT_DIR / "tables" / "book_summary.csv"
summary_df.to_csv(summary_path, index=False)
print(f"wrote {summary_path}: {len(summary_df):,} rows")

manifest = {
    "n_volumes": len(docs),
    "sentences_before": SENTENCES_BEFORE,
    "sentences_after": SENTENCES_AFTER,
    "flag_threshold_pct": FLAG_THRESHOLD_PCT,
    "shuffle_seed": SHUFFLE_SEED,
    "env_lexicon_size": len(env_words),
    "tech_lexicon_size": len(tech_words),
    "env_passages": len(env_rows),
    "tech_passages": len(tech_rows),
    "n_flagged": len(flagged_books),
}
manifest_path = OUT_DIR / "MANIFEST.json"
manifest_path.write_text(json.dumps(manifest, indent=2))
print(f"wrote {manifest_path}")

## Verify every file actually landed on disk

Re-reads each file fresh from disk and checks it against the in-memory data -- this is
the check that was missing before.

In [ ]:
def verify_csv_parts(parts, expected_total_rows, label):
    total = 0
    for p in parts:
        assert p.exists(), f"MISSING: {p} was supposed to be written but does not exist on disk"
        total += len(pd.read_csv(p))
    assert total == expected_total_rows, (
        f"MISMATCH for {label}: wrote {expected_total_rows} rows but disk shows {total}"
    )
    print(f"VERIFIED: {label} has {total:,} rows on disk across {len(parts)} file(s), matches expected {expected_total_rows:,}.")


verify_csv_parts(env_parts, len(env_rows), "environment_context")
verify_csv_parts(tech_parts, len(tech_rows), "technology_context")

reread_summary = pd.read_csv(summary_path)
assert len(reread_summary) == len(summary_df), (
    f"MISMATCH: book_summary.csv wrote {len(summary_df)} rows but disk shows {len(reread_summary)}"
)
print(f"VERIFIED: {summary_path} has {len(reread_summary):,} rows on disk, matches expected {len(summary_df):,}.")

reread_manifest = json.loads(manifest_path.read_text())
assert reread_manifest == manifest, "MISMATCH: manifest on disk does not match what was written"
print(f"VERIFIED: {manifest_path} matches on disk.")

## Release

Only run this once every cell above has printed `VERIFIED`, and after reviewing the
flagged-books list above. `add` and `done` are kept in separate cells so you can read
the `add` output before committing to `done`.

In [ ]:
import subprocess


def run_releaseresults(*args):
    cmd = ["releaseresults", *args]
    print("$ " + " ".join(cmd))
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(f"releaseresults exited with code {result.returncode}")
    return result

In [ ]:
run_releaseresults("add", str(OUT_DIR / "tables"),
                    *[str(p) for p in env_parts], *[str(p) for p in tech_parts],
                    str(OUT_DIR / "MANIFEST.json"))

Check the output above looks right, then run this to finalize:

In [ ]:
run_releaseresults("done")